# City data from Wikipedia

Scrape a few cities from Wikipedia (location, country, population) and save them into the `gans` database. Run this one first, the weather and flight notebooks read the `city` table.

In [1]:
# %pip install lat-long-parser
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import sqlalchemy 
import pymysql 
from getpass import getpass
from lat_lon_parser import parse    # for decimal coordinates
import os
from dotenv import load_dotenv

### Setup
Display settings, the MySQL connection and the RapidAPI key.

In [2]:
def configure_pandas_display():
    """Show full DataFrames without truncation."""

    pd.set_option('display.max_rows', None)      # show all rows
    pd.set_option('display.max_columns', None)   # show all columns
    pd.set_option('display.max_colwidth', None)  # don't cut off long cell text
    pd.set_option('display.width', None)         # use the full terminal/notebook width

In [3]:
def build_connection_string(schema="gans", host="127.0.0.1", user="root", port=3306):
    """Ask for the MySQL password and return a SQLAlchemy connection string."""

    # the name of the MySQL database to connect to
    # the server address — 127.0.0.1 means "this same computer" (localhost)
    # the MySQL username to log in with
    # the network port MySQL listens on (3306 is MySQL's default)

    password = getpass('Enter MySQL password: ')      # the password for that MySQL user

    # Build the full connection URL that SQLAlchemy/pymysql needs to connect:
    # format: mysql+pymysql://<user>:<password>@<host>:<port>/<schema>
    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    return connection_string

In [4]:
def load_api_key():
    """Read the RapidAPI key from the .env file and return it (None if missing)."""
    load_dotenv()  # reads .env into environment variables
    api_key = os.getenv("rapid_api_key")
    if api_key:
        print("rapid_api_key is set")
        return api_key
    else:
        print('rapid_api_key is not set')

### Scraping
`info()` pulls the details for one city, `collect_cities()` loops over the list and builds the two dataframes.

In [5]:
def info(city = 'Berlin'):
    """Scrape one city's Wikipedia page for its details."""
    # build the Wikipedia URL and request the page
    url = 'https://en.wikipedia.org/wiki/' + city
    headers = {'User-Agent': 'Chrome/134.0.0.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    # page title, coordinates and country from the infobox
    city = soup.find(class_='mw-page-title-main').get_text()
    latitude = soup.find_all(class_ = 'latitude')[0].get_text()
    longitude = soup.find_all(class_ = 'longitude')[0].get_text()
    country = soup.find_all(class_='infobox-data')[0].get_text()
 
    # Find the "Population" section, then pull its year and total from the
    # row immediately following the header
    population_headers = soup.find_all('th', class_='infobox-header')
    population = None
    year = None
    for header in population_headers:
        if header.get_text(strip=True).startswith('Population'):
            year = re.search(r'\d{4}', header.get_text(strip=True)).group()
            # Goes up from the <th> to find the <tr> that wraps it.
            row = header.find_parent('tr')
            # Moves sideways to the very next <tr> in the table.
            next_row = row.find_next_sibling('tr') 
            # Inside that row, searches for the <td> tag specifically for data     
            population = next_row.find('td').get_text(strip=True)
            break
        
    return city, latitude, longitude, country, population, year

In [6]:
def collect_cities(cities):
    """Scrape every city and return the city and population DataFrames."""
    city_d = {}
    population_d = {}

    for i in cities:
        print(i)
        city, latitude, longitude, country, population, year = info(i)
        city_d[i] = {
            'latitude': parse(latitude),
            'longitude': parse(longitude),
            'country': country
        }
        population_d[i] = {
             'population': population,
             'year': year       
        }

    city_df = pd.DataFrame(city_d).T
    city_df = city_df.reset_index().rename(columns={'index': 'name'})

    population_df = pd.DataFrame(population_d).T
    population_df = population_df.reset_index().rename(columns={'index':'name'})
    return city_df, population_df

### Save to MySQL
Cities go in first so they get a `city_id`, then population is matched to them by name.

In [7]:
def save_cities(city_df, connection_string):
    """Append city rows to the city table."""
    return city_df.to_sql(
        'city',                    # the name of the target table in the database
        if_exists='append',        # how to behave if the table already exists 
                                   # append : adds your DataFrame's rows onto the end of the existing table, keeping whatever's already there.
                                   # replace : drop the entire table and recreate it from scratch.
        con=connection_string,     # the database connection to write through
        index=False                # don't write the DataFrame's row index as its own column
    )

In [8]:
def save_population(population_df, connection_string):
    """Look up city_ids by name and append population rows to the population table."""
    city_lookup = pd.read_sql('SELECT city_id, name FROM city', con=connection_string)
    population_df_final = population_df.merge(city_lookup, on='name', how='left')
    return population_df_final.to_sql('population',
                      if_exists='append',
                      con=connection_string,
                      index=False)

### Airports
Airports within 50 km of each city, read back from the `city` table and saved to `city_icao`.

In [9]:
def get_cities(connection_string):
    """Load city ids and coordinates from the database."""
    return pd.read_sql(
        'SELECT city_id, name, latitude, longitude FROM city',
        con=connection_string,
    )

In [10]:
def fetch_nearby_airports(city_id, lat, lon, api_key):
    """Find airports within 50 km of one city as a list of dicts."""
    response = requests.get(
        "https://aerodatabox.p.rapidapi.com/airports/search/location",
        headers={"x-rapidapi-key": api_key, "x-rapidapi-host": "aerodatabox.p.rapidapi.com"},
        params={"withFlightInfoOnly": True, "radiusKm": 50, "limit": 10, "lat": lat, "lon": lon},
    )
    if response.status_code == 204:   # no airports near this city
        return []
    response.raise_for_status()   # stop with a clear error if the request fails

    rows = []
    for item in response.json()["items"]:
        rows.append({
            "city_id": city_id,
            "icao": item["icao"],
            "name": item["name"],
        })
    return rows

In [11]:
def collect_airports(city_df, api_key):
    """Fetch nearby airports for every city and return one DataFrame."""
    records = []
    for _, city in city_df.iterrows():
        records.extend(fetch_nearby_airports(
            city['city_id'], city['latitude'], city['longitude'], api_key
        ))
    return pd.DataFrame(records)

In [12]:
def save_airports(airport_df, connection_string):
    """Append the airports to the city_icao table."""
    return airport_df.to_sql(
        name='city_icao', con=connection_string,
        if_exists='append', index=False,
    )

### Run

In [13]:
def run_city_pipeline(cities):
    """Run the whole flow: scrape cities, save city and population rows, find and save nearby airports."""
    configure_pandas_display()
    city_df, population_df = collect_cities(cities)
    connection_string = build_connection_string()
    save_cities(city_df, connection_string)
    save_population(population_df, connection_string)
    city_dff = get_cities(connection_string)
    
    airport_df = collect_airports(city_dff, load_api_key())
    save_airports(airport_df, connection_string)
    return city_df, population_df, airport_df

In [14]:
# cities = [
#     'Berlin', 'Hamburg', 'New_Delhi', 'Frankfurt', 'Stuttgart', 'Mumbai', 'Mathura',
#     'Paris', 'London', 'Rome', 'Madrid', 'Vienna', 'Prague', 'Amsterdam',
#     'Dublin', 'Brussels', 'Warsaw', 'Budapest', 'Athens', 'Lisbon',
#     'Moscow', 'Istanbul', 'Cairo', 'Toronto', 'Vancouver', 'Chicago',
#     'Tokyo', 'Beijing', 'Shanghai', 'Seoul',
#     'Bangkok', 'Jakarta', 'Manila', 'Dubai', 'Chennai',
#     'Bangalore', 'Hyderabad', 'Jaipur', 'Lucknow', 'Nairobi', 'Lagos'
# ]

cities = [
    'Berlin', 'Hamburg', 'New_Delhi'
]

# Melbourne, Sydney, Kolkata, Paris [NULL Population]

In [15]:
city_df, population_df, airport_df = run_city_pipeline(cities)
airport_df

Berlin
Hamburg
New_Delhi
rapid_api_key is set


,city_id,icao,name
0,1,EDDT,Berlin -Tegel
1,1,EDDB,Berlin Brandenburg
2,2,EDDH,Hamburg
3,3,VIDP,New Delhi Indira Gandhi
4,3,VIDX,Ghaziabad Hindon
5,4,EDDT,Berlin -Tegel
6,4,EDDB,Berlin Brandenburg
7,5,EDDH,Hamburg
8,6,VIDP,New Delhi Indira Gandhi
9,6,VIDX,Ghaziabad Hindon
